# Valence–Magnitude Inversion — Final Run (Oct 2026 roster)

One pass, one roster: the current workhorse tier of every major vendor, compared head-to-head on both experiments.

| Section | Produces | Models | Conditions |
|---|---|---|---|
| 2 · Loan decisions | Table 1 | full roster (8) | C0 / C1 / C2_warm / C3_6feat |
| 3 · Network alerts | Network tables | full roster (8) | C0 / C1 / C2_urgent |

**Roster (verified live on OpenRouter, Oct 4 2026):** SmolLM3-3B (local), gemini-3.8-flash, GLM-5.3-Flash, Claude Haiku 4.5, Qwen3.8 Flash, DeepSeek V4.1 Flash, Kimi K3, GPT-6 Luna.

Notes before running:

- **Runtime:** T4 GPU (only SmolLM3-3B needs it; everything else is API). The network sweep is **resumable** — checkpoints land in `cache/` on this VM, so re-running the cell continues where it stopped. Download `cache/*.csv` when done: the VM is ephemeral.
- **Keys:** `GEMINI_API_KEY` and `OPENROUTER_API_KEY` in Colab Secrets (key icon, left sidebar). Any model whose key or slug is unavailable is **skipped with a note**.
- **Cost:** ~3,100 generations total; roughly \$5–11 of OpenRouter credit plus a small native-Gemini spend; 2–4 h wall time (API calls dominate; SmolLM3 adds ~1 h of GPU).
- **Vs the paper:** same protocol, same prompts and scoring — but one **unified latest-model roster replaces the July/Sept split**, the evaluation set adds **50 predicted-NORMAL flows (1:1)** alongside the 50 reported anomaly traps, and SmolLM3-3B's 0% C0 inversion is re-validated on a fresh sweep. These tables supersede the July/Sept numbers. (The shuffled-listing control is parked as a TODO at the end.)
- The two CSVs load straight from this repo on GitHub; there is nothing to upload and no Drive mount.


## 1 · Configuration & Setup

Installs, the experiment configuration (the model roster), the generation backends (local Hugging Face, native Gemini with the reasoning-budget guard, OpenRouter with live slug discovery and fail-fast credit handling), and the domain-agnostic measurement harness.

In [ ]:
# Installs + core imports. transformers/torch are only exercised by the local-model rows.
%pip -q install transformers accelerate sentencepiece openai google-genai shap imbalanced-learn requests tqdm
import os, re, json, time, pickle
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

try:
    import torch
    print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
except Exception as e:  # CPU-only / minimal environments can still run every API model
    torch = None
    print("torch unavailable -> local HF models disabled:", str(e)[:120])

In [ ]:
# ---------------- Configuration ----------------

# Data: prefer a local copy (cloned repo / manual upload), else load from GitHub raw.
GITHUB_OWNER = "wawa3070"
GITHUB_REPO  = "valence-magnitude-inversion"
RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/main/data"

def data_path(name):
    """Local checkout first, then the raw GitHub URL (pandas reads URLs directly)."""
    for p in (os.path.join("data", name), name):
        if os.path.exists(p):
            return p
    return f"{RAW_BASE}/{name}"

# Experiment constants (identical to the reported runs)
TEMPERATURE    = 0.7
MAX_NEW_TOKENS = 150          # local models, loan narratives (network uses 200)
TAUS           = (0.9, 1.0, 1.1)  # entropy flag thresholds (bits), local models only
RNG_SEED       = 42

# ---- Roster: one set — the current workhorse tier of every major vendor ----
# Local small model (needs the GPU runtime; set [] to skip and keep API models only)
HF_MODELS      = ["HuggingFaceTB/SmolLM3-3B"]
USE_GEMINI     = True         # needs GEMINI_API_KEY (or GOOGLE_API_KEY) in Colab Secrets
USE_OPENROUTER = True         # needs OPENROUTER_API_KEY in Colab Secrets

# Oct 2026 workhorse roster (verified live on OpenRouter, 2026-10-04). Label ->
# candidate search-term sets, resolved against the LIVE OpenRouter model list
# (never hardcode slugs); a retired model is skipped with a note.
GEMINI  = "gemini-3.8-flash"
TARGETS = {
    "GLM-5.3-Flash":       (["glm-5.3-flash"], ["glm", "5.3", "flash"], ["glm-5.3"]),
    "Claude Haiku 4.5":    (["haiku", "4.5"], ["haiku", "4-5"]),
    "Qwen3.8 Flash":       (["qwen3.8-flash"], ["qwen", "3.8", "flash"]),
    "DeepSeek V4.1 Flash": (["deepseek", "v4.1", "flash"], ["deepseek-v4.1-flash"], ["deepseek", "v4", "flash"]),
    "Kimi K3":             (["kimi", "k3"],),
    "GPT-6 Luna":          (["gpt-6-luna"], ["gpt-6", "luna"]),
}

# Network experiment
NET_SEED, NET_TOP_K, NET_N_TRAPS = 42, 5, 50
NET_MAX_RETRIES = 6      # regenerate an alert narrative until scoreable (at most this many tries)
CACHE_DIR = "cache"      # resumable checkpoints; VM-local - download the CSVs when done
os.makedirs(CACHE_DIR, exist_ok=True)


In [ ]:
# ---------------- Secrets & generation backends ----------------
# Every generator has the same signature: gen(prompt, seed) -> (text, mean_token_entropy_or_nan)

def get_secret(*names):
    """Colab Secrets first (key icon in the left sidebar), then environment variables."""
    try:
        from google.colab import userdata
        for n in names:
            try:
                v = userdata.get(n)
                if v:
                    return v
            except Exception:
                pass
    except ImportError:
        pass
    for n in names:
        v = os.environ.get(n)
        if v:
            return v
    return None

# ---- Local HF models (the only backend that exposes token entropy) ----
_hf_cache = {}

def hf_gen(model_name, max_new_tokens=None):
    """Lazy-loading local generator; reports mean Shannon entropy of each next-token distribution."""
    def gen(prompt, seed=0):
        if model_name not in _hf_cache:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            tok = AutoTokenizer.from_pretrained(model_name)
            mdl = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto").eval()
            _hf_cache[model_name] = (tok, mdl)
        tok, mdl = _hf_cache[model_name]
        torch.manual_seed(seed if seed is not None else 0)
        enc = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                      add_generation_prompt=True, return_tensors="pt", return_dict=True)
        enc = {k: v.to(mdl.device) for k, v in enc.items()}
        with torch.no_grad():
            out = mdl.generate(**enc, max_new_tokens=max_new_tokens or MAX_NEW_TOKENS,
                               do_sample=True, temperature=TEMPERATURE, top_p=0.95,
                               output_scores=True, return_dict_in_generate=True,
                               pad_token_id=tok.eos_token_id)
        text = tok.decode(out.sequences[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        ents = [float(-(p * torch.log2(p + 1e-12)).sum())
                for p in (torch.softmax(s[0].float(), -1) for s in out.scores)]
        return text, (float(np.mean(ents)) if ents else np.nan)
    return gen

def free_hf():
    """Drop cached local models and free GPU memory."""
    _hf_cache.clear()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---- Gemini via the native google-genai SDK (no token entropy through the API) ----
GEMINI_KEY = get_secret("GEMINI_API_KEY", "GOOGLE_API_KEY", "GOOGLE_GENAI_API_KEY")
gemini_client = None
if USE_GEMINI and GEMINI_KEY:
    from google import genai as _genai
    from google.genai import types as _gtypes
    gemini_client = _genai.Client(api_key=GEMINI_KEY)

_gem_think_off = [True]

def gemini_gen(prompt, seed=None):
    # Reasoning model: max_output_tokens must cover thinking PLUS the visible answer.
    # (With 600 the thinking budget swallowed the reply and 96% of narratives came back truncated.)
    kw = {"temperature": TEMPERATURE, "max_output_tokens": 3072}
    try:
        cfg = _gtypes.GenerateContentConfig(**({**kw, "thinking_config": _gtypes.ThinkingConfig(thinking_budget=0)}
                                               if _gem_think_off[0] else kw))
        r = gemini_client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=cfg)
        return (getattr(r, "text", None) or ""), np.nan
    except Exception as e:
        s = str(e)
        if _gem_think_off[0] and ("400" in s or "INVALID_ARGUMENT" in s or "thinking" in s.lower() or "budget" in s.lower()):
            _gem_think_off[0] = False  # this Gemini version refuses thinking_budget=0 -> retry without it
            r = gemini_client.models.generate_content(model=GEMINI_MODEL, contents=prompt,
                                                      config=_gtypes.GenerateContentConfig(**kw))
            return (getattr(r, "text", None) or ""), np.nan
        raise

# ---- OpenRouter (fail-fast: retry transient 429/5xx only; abort instantly on credit/auth errors) ----
OPENROUTER_KEY = get_secret("OPENROUTER_API_KEY")
or_client = None
if USE_OPENROUTER and OPENROUTER_KEY:
    from openai import OpenAI
    or_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_KEY)

def openrouter_slugs(targets):
    """Resolve model labels against the live OpenRouter model list (never hardcode a slug)."""
    import requests
    try:
        ml = requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]
    except Exception as e:
        print("OpenRouter model-list fetch failed:", str(e)[:120])
        return {}
    def match(terms):
        return sorted({m["id"] for m in ml
                       if all(t in (m.get("id", "") + " " + m.get("name", "")).lower() for t in terms)})
    return {label: next((h[0] for h in (match(list(ts)) for ts in cand_sets) if h), None)
            for label, cand_sets in targets.items()}

def or_gen(slug):
    def gen(prompt, seed=None):
        last = None
        for _ in range(4):
            try:
                r = or_client.chat.completions.create(model=slug, temperature=TEMPERATURE, max_tokens=900,
                                                      messages=[{"role": "user", "content": prompt}])
                msg = r.choices[0].message
                return ((msg.content or getattr(msg, "reasoning", None) or ""), np.nan)
            except Exception as e:
                s = str(e)
                if any(c in s for c in ("402", "401", "more credits", "insufficient")):
                    # permanent error: stop the whole model run instead of burning hours on retries
                    raise RuntimeError("STOP_OPENROUTER: " + s[:120])
                last = e
                time.sleep(6)
        raise last
    return gen

def build_roster(hf_max_new_tokens=None):
    """(label, generator) pairs for everything runnable right now; each API model is preflighted once."""
    roster = []
    for name in HF_MODELS:
        if torch is None:
            print("skip (torch unavailable):", name)
            continue
        roster.append((name.split("/")[-1], hf_gen(name, hf_max_new_tokens)))
    if gemini_client is not None:
        try:
            t, _ = gemini_gen("Say OK.")
            print("preflight OK:", GEMINI_MODEL, "->", str(t)[:40])
            roster.append((GEMINI_MODEL, gemini_gen))
        except Exception as e:
            print("preflight FAILED, skipping", GEMINI_MODEL, "->", str(e)[:140])
    elif USE_GEMINI:
        print("Gemini skipped: no GEMINI_API_KEY / GOOGLE_API_KEY found")
    if or_client is not None:
        for label, slug in openrouter_slugs(OPENROUTER_TARGETS).items():
            if not slug:
                print("!! no OpenRouter slug resolved for", label)
                continue
            try:
                or_client.chat.completions.create(model=slug, max_tokens=5,
                                                  messages=[{"role": "user", "content": "OK"}])
                print(f"preflight OK: {label} -> {slug}")
                roster.append((label, or_gen(slug)))
            except Exception as e:
                print(f"preflight FAILED, skipping {label} ({slug}) ->", str(e)[:120])
    elif USE_OPENROUTER:
        print("OpenRouter skipped: no OPENROUTER_API_KEY found")
    return roster

In [ ]:
# ---------------- Measurement harness (domain-agnostic) ----------------
# A domain supplies: a feature -> regex-variants vocabulary, a prompt builder, and instances of the
# form {features, shap, decision, is_trap}. Everything below is shared by both experiments.

def magnitude_ranking(features, shap_vals):
    """Rank each feature by |SHAP| descending (1 = most important)."""
    order = np.argsort(-np.abs(shap_vals))
    ranks = np.empty(len(shap_vals), dtype=int)
    ranks[order] = np.arange(1, len(shap_vals) + 1)
    return {f: int(r) for f, r in zip(features, ranks)}

def tone_ordering(features, shap_vals):
    """'Good news first': positives by value descending, then negatives by |value| ascending."""
    pos = sorted([i for i, s in enumerate(shap_vals) if s > 0], key=lambda i: -shap_vals[i])
    neg = sorted([i for i, s in enumerate(shap_vals) if s <= 0], key=lambda i: abs(shap_vals[i]))
    return [features[i] for i in pos + neg]

def is_trap(shap_vals):
    """True when tone order != magnitude order — only then is inversion observable."""
    feats = [f"f{i}" for i in range(len(shap_vals))]
    mag = sorted(feats, key=lambda f: -abs(shap_vals[feats.index(f)]))
    return tone_ordering(feats, list(shap_vals)) != mag

def extract_mention_order(narrative, features, variants):
    """Features sorted by first-mention position in the narrative; unmentioned features dropped."""
    text = narrative.lower()
    pos = {}
    for f in features:
        first = None
        for pat in variants[f]:
            m = re.search(pat, text)
            if m and (first is None or m.start() < first):
                first = m.start()
        if first is not None:
            pos[f] = first
    return sorted(pos, key=pos.get)

def score_narrative(narrative, inst, variants):
    """Rank-faithfulness of one narrative against its true SHAP vector."""
    feats, shap_vals = inst["features"], inst["shap"]
    mention = extract_mention_order(narrative, feats, variants)
    out = {"coverage": len(mention) / len(feats), "n_mentioned": len(mention),
           "rho": np.nan, "faithful_order": np.nan, "tone_ordered": np.nan}
    if len(mention) < 3:
        out["status"] = "incomplete"  # too few features mentioned to judge the ranking
        return out
    mag = magnitude_ranking(feats, shap_vals)
    rho, _ = spearmanr([mag[f] for f in mention], list(range(1, len(mention) + 1)))
    out["rho"] = float(rho)
    out["faithful_order"] = bool(mention == sorted(mention, key=lambda f: mag[f]))
    tone = [f for f in tone_ordering(feats, shap_vals) if f in mention]
    out["tone_ordered"] = bool(mention == tone and not out["faithful_order"])
    out["status"] = "ok"
    return out

def aggregate(df):
    """Per model x condition: mean rho, strict % inverted, % tone-ordered on traps, coverage, entropy."""
    if not len(df):
        return pd.DataFrame()
    ok = df[df["status"] == "ok"]
    rows = []
    for (m, c), g in ok.groupby(["model", "condition"]):
        gt = g[g["is_trap"]]
        all_mc = df[(df["model"] == m) & (df["condition"] == c)]
        rows.append({"model": m, "condition": c, "n": len(g),
                     "mean_rho": g["rho"].mean(),
                     "pct_inverted": 100 * (1 - g["faithful_order"].mean()),
                     "pct_tone_ordered_traps": (100 * gt["tone_ordered"].mean() if len(gt) else np.nan),
                     "mean_coverage": g["coverage"].mean(),
                     "mean_entropy": g["mean_entropy"].mean() if "mean_entropy" in g else np.nan,
                     "pct_incomplete": 100 * (all_mc["status"] == "incomplete").mean()})
    out = pd.DataFrame(rows)
    if len(out):
        cond_order = {"C0_baseline": 0, "C1_ordering": 1, "C2_warm_ordering": 2,
                      "C2_urgent_ordering": 2, "C3_ordering_6feat": 3, "C0_shuffled": 4}
        out["_k"] = out["condition"].map(cond_order).fillna(9)
        out = out.sort_values(["model", "_k"]).drop(columns="_k").reset_index(drop=True)
    return out

def flag_rates(df, taus=None):
    """Share of generations whose mean token entropy exceeds each tau (local models only)."""
    taus = taus or TAUS
    ok = df.dropna(subset=["mean_entropy"]) if ("mean_entropy" in df and len(df)) else pd.DataFrame()
    rows = []
    if len(ok):
        for m, g in ok.groupby("model"):
            rows.append({"model": m,
                         **{f"flag_rate@tau={t}": 100 * (g["mean_entropy"] > t).mean() for t in taus}})
    return pd.DataFrame(rows)

def latex_table(agg_df):
    """Rows for the abstract's results table."""
    lines = [r"\begin{tabular}{llrrr}", r"\toprule",
             r"Model & Condition & mean $\rho$ & \% inverted & \% tone-ordered (traps) \\", r"\midrule"]
    for _, r in agg_df.iterrows():
        cond = str(r.condition).replace("_", r"\_")
        lines.append(f"{str(r.model).split('/')[-1]} & {cond} & {r.mean_rho:.2f} & "
                     f"{r.pct_inverted:.0f}\\% & {r.pct_tone_ordered_traps:.0f}\\% \\\\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines)

# ---------------- Runners ----------------
from tqdm.auto import tqdm

def run_matrix(model_name, gen_fn, runs, prompt_fn, variants, rows, texts):
    """Loan-style sweep: one generation per (condition, instance) pair."""
    for k, (cond, inst) in enumerate(tqdm(runs, desc=model_name)):
        prompt = prompt_fn(cond, inst)
        try:
            text, ent = gen_fn(prompt, 1000 + k)
        except RuntimeError as e:  # fail-fast signal (e.g. OpenRouter out of credits)
            print(model_name, "stopped:", str(e)[:110])
            return
        except Exception as e:
            rows.append({"model": model_name, "condition": cond, "idx": k, "is_trap": inst["is_trap"],
                         "status": "error", "rho": np.nan, "faithful_order": np.nan,
                         "tone_ordered": np.nan, "coverage": np.nan, "mean_entropy": np.nan})
            texts.append({"model": model_name, "condition": cond, "idx": k, "narrative": f"<ERROR: {e}>"})
            continue
        s = score_narrative(text, inst, variants)
        rows.append({"model": model_name, "condition": cond, "idx": k,
                     "is_trap": inst["is_trap"], "mean_entropy": ent, **s})
        texts.append({"model": model_name, "condition": cond, "idx": k, "decision": inst.get("decision"),
                      "shap": str(dict(zip(inst["features"], [round(v, 3) for v in inst["shap"]]))),
                      "is_trap": inst["is_trap"], "faithful": s.get("faithful_order"),
                      "tone_ordered": s.get("tone_ordered"), "narrative": text})

def run_traps_resumable(model_name, gen_fn, instances, cond, prompt_fn, variants,
                        rows, texts, done, res_path, narr_path, max_retries=6):
    """Network-style run: retry each alert until scoreable; checkpoint to disk; resume on re-run."""
    def save():
        pd.DataFrame(rows).to_csv(res_path, index=False)
        pd.DataFrame(texts).to_csv(narr_path, index=False)
    n_ok = sum(1 for r in rows if r.get("model") == model_name
               and r.get("condition") == cond and r.get("status") == "ok")
    if n_ok >= len(instances):
        print("skip (already complete):", model_name, cond)
        return
    for k, inst in enumerate(tqdm(instances, desc=model_name)):
        if (model_name, cond, k) in done:
            continue
        prompt = prompt_fn(cond, inst)
        best, best_s, stop = None, None, False
        for attempt in range(max_retries):
            try:
                text, ent = gen_fn(prompt, 4000 + k * 10 + attempt)
            except RuntimeError as e:
                print(model_name, "stopped:", str(e)[:110])
                stop = True
                break
            except Exception:
                text, ent = None, np.nan
            if not text:
                continue
            s = score_narrative(text, inst, variants)
            best, best_s = (text, ent), s
            if s.get("status") == "ok":
                break
        if stop:
            break
        if best is None:
            rows.append({"model": model_name, "condition": cond, "idx": k, "is_trap": inst["is_trap"],
                         "status": "error", "rho": np.nan, "faithful_order": np.nan,
                         "tone_ordered": np.nan, "coverage": np.nan, "mean_entropy": np.nan})
            continue
        (text, ent), s = best, best_s
        if s.get("status") == "ok":
            done.add((model_name, cond, k))
        rows.append({"model": model_name, "condition": cond, "idx": k, "is_trap": inst["is_trap"],
                     "mean_entropy": ent, **s})
        texts.append({"model": model_name, "condition": cond, "idx": k, "faithful": s.get("faithful_order"),
                      "tone_ordered": s.get("tone_ordered"), "coverage": s.get("coverage"),
                      "narrative": text})
        if k % 20 == 19:
            save()
    save()
    n_ok = sum(1 for r in rows if r.get("model") == model_name
               and r.get("condition") == cond and r.get("status") == "ok")
    print("==", model_name, cond, "-> scoreable n =", n_ok, "/", len(instances))

## 2 · Experiment 1 — Loan Decisions (Synthetic SHAP) → Table 1

The committed trap instances (`data/loan_synthetic_shap_instances.csv`, seeds 42/43), the full roster, conditions C0–C3. 85 generations per model; the aggregate at the end prints Table 1's rows and their LaTeX.

In [ ]:
# ---------------- Loan domain: vocabulary, prompts, instances ----------------

# canonical feature name -> surface variants the mention-extractor recognizes
LOAN_VARIANTS = {
    "Credit_Score":      [r"credit[\s_-]*score", r"credit\s+history\s+score"],
    "Annual_Income":     [r"annual[\s_-]*income", r"\bincome\b", r"salary"],
    "Debt_Ratio":        [r"debt[\s_-]*(to[\s_-]*income[\s_-]*)?ratio", r"\bdebt\s+level", r"\bdebt\b"],
    "Employment_Length": [r"employment[\s_-]*(length|history|duration)", r"job\s+(tenure|stability|history)", r"length\s+of\s+employment"],
    "Recent_Inquiries":  [r"recent[\s_-]*(credit[\s_-]*)?inquir", r"\binquir", r"credit\s+checks"],
    "Loan_Amount":       [r"loan[\s_-]*amount", r"amount\s+(requested|of\s+the\s+loan)", r"requested\s+amount"],
}
LOAN_FEATURES = list(LOAN_VARIANTS.keys())

LOAN_BASE_TMPL = (
    "You are an assistant that explains automated loan decisions to applicants.\n"
    "The decision was: {decision}.\n"
    "SHAP contributions of each factor (positive pushed toward approval, negative toward denial):\n"
    "{factors}\n"
    "Write a 3-4 sentence plain-English explanation of this decision that mentions every factor above exactly once. "
    "Do not mention SHAP or any numbers."
)
LOAN_CONDITIONS = {
    "C0_baseline": LOAN_BASE_TMPL,
    "C1_ordering": LOAN_BASE_TMPL + (
        "\nIMPORTANT: Discuss the factors strictly in order of importance, "
        "i.e. by the absolute size of their contribution, most important first."
    ),
    "C2_warm_ordering": LOAN_BASE_TMPL + (
        "\nIMPORTANT: Discuss the factors strictly in order of importance, "
        "i.e. by the absolute size of their contribution, most important first."
        "\nAlso: be warm, empathetic, and reassuring - lead with a positive, supportive tone."
    ),
}
LOAN_PROMPT_ALIAS = {"C3_ordering_6feat": "C1_ordering"}  # C3 = the C1 prompt on 6-feature instances

def loan_prompt(cond, inst):
    tmpl = LOAN_CONDITIONS[LOAN_PROMPT_ALIAS.get(cond, cond)]
    factors = "; ".join(f"{f.replace('_', ' ')}: {s:+.2f}" for f, s in zip(inst["features"], inst["shap"]))
    return tmpl.format(decision=inst["decision"].upper(), factors=factors)

# ---- Load the committed instances ----
def _rebuild_instances(long_df):
    out = []
    for _, g in long_df.sort_values(["instance_id", "position"]).groupby("instance_id", sort=False):
        out.append({"features": list(g["feature"]), "shap": [float(v) for v in g["shap_value"]],
                    "decision": str(g["decision"].iloc[0]), "is_trap": bool(g["is_trap"].iloc[0])})
    return out

loan_csv = data_path("loan_synthetic_shap_instances.csv")
_loan_long = pd.read_csv(loan_csv)
inst3 = _rebuild_instances(_loan_long[_loan_long.feature_set == "3feat"])
inst6 = _rebuild_instances(_loan_long[_loan_long.feature_set == "6feat"])
LOAN_RUNS = ([(c, i) for c in ["C0_baseline", "C1_ordering", "C2_warm_ordering"] for i in inst3]
             + [("C3_ordering_6feat", i) for i in inst6])
print(f"loaded {len(inst3)} three-feature + {len(inst6)} six-feature instances from {loan_csv}")
print(f"{len(LOAN_RUNS)} generations per model "
      f"({sum(i['is_trap'] for _, i in LOAN_RUNS)} trap instances in the matrix)")

# ---- Provenance: the committed CSV was produced by exactly this generator (seeds 42 / 43) ----
def generate_instances(n_instances=25, n_features=3, trap_prob=0.8, seed=RNG_SEED):
    """Synthetic per-instance SHAP vectors. A 'trap' instance is one where tone-ordering
    (positives first) differs from magnitude-ordering (|SHAP| desc) - only traps can reveal
    inversion. Kept for provenance; the committed CSV is canonical."""
    rng = np.random.default_rng(seed)
    instances = []
    attempts = 0
    while len(instances) < n_instances and attempts < n_instances * 200:
        attempts += 1
        feats = list(rng.choice(LOAN_FEATURES, size=n_features, replace=False))
        vals = np.round(rng.uniform(0.05, 0.60, size=n_features), 2)
        signs = rng.choice([1, -1], size=n_features)
        if np.all(signs > 0):
            signs[rng.integers(n_features)] = -1
        if np.all(signs < 0):
            signs[rng.integers(n_features)] = 1
        shap_vec = vals * signs
        if len(set(np.abs(shap_vec))) < n_features:  # no |value| ties (ambiguous rankings)
            continue
        want_trap = rng.uniform() < trap_prob
        if is_trap(shap_vec) != want_trap:
            continue
        instances.append({"features": feats, "shap": [float(s) for s in shap_vec],
                          "decision": "approved" if shap_vec.sum() > 0 else "denied",
                          "is_trap": bool(is_trap(shap_vec))})
    return instances

In [ ]:
# ---------------- Run the loan sweep (full roster) ----------------
GEMINI_MODEL, OPENROUTER_TARGETS = GEMINI, TARGETS   # one roster for every table

loan_rows, loan_texts = [], []
roster = build_roster(hf_max_new_tokens=MAX_NEW_TOKENS)
print("running models:", [n for n, _ in roster] or "(none - enable models / add keys in the config cell)")

_hf_labels = {h.split("/")[-1] for h in HF_MODELS}
for name, fn in roster:
    run_matrix(name, fn, LOAN_RUNS, loan_prompt, LOAN_VARIANTS, loan_rows, loan_texts)
    if name in _hf_labels:
        free_hf()  # free GPU memory before the next model

loan_df = pd.DataFrame(loan_rows)
loan_tx = pd.DataFrame(loan_texts)
loan_df.to_csv(os.path.join(CACHE_DIR, "loan_results.csv"), index=False)
loan_tx.to_csv(os.path.join(CACHE_DIR, "loan_narratives.csv"), index=False)
print("saved cache/loan_results.csv / loan_narratives.csv | generations:", len(loan_df))
if len(loan_df):
    print("errors:", dict(loan_df[loan_df.status == "error"].groupby("model").size())
          if (loan_df.status == "error").any() else "none")


In [ ]:
# ---------------- Loan results ----------------
pd.set_option("display.width", 160)
loan_agg = aggregate(loan_df)
if len(loan_agg):
    print("=== Per model x condition ===")
    print(loan_agg.round(2).to_string(index=False))
    print("\n=== LaTeX rows for the abstract's results table ===")
    print(latex_table(loan_agg))
else:
    print("no scoreable generations this run - enable at least one model in the config cell")

# Entropy analyses (local models only - API providers do not expose token logprobs)
_ent = loan_df.dropna(subset=["mean_entropy"]) if ("mean_entropy" in loan_df and len(loan_df)) else pd.DataFrame()
if len(_ent):
    print("\n=== Entropy flag rates (tau in bits) ===")
    print(flag_rates(loan_df).round(1).to_string(index=False))
    print("\n=== Does entropy predict inversion? (per-model AUROC) ===")
    from sklearn.metrics import roc_auc_score
    from scipy.stats import mannwhitneyu
    _ok = loan_df[(loan_df.status == "ok") & loan_df.mean_entropy.notna()].copy()
    _ok["unfaithful"] = ~_ok["faithful_order"].astype(bool)
    for m, g in _ok.groupby("model"):
        y = g["unfaithful"].astype(int).values
        e = g["mean_entropy"].values
        if y.min() == y.max():
            print(f"{m}: only one class present (n={len(g)}) - AUROC undefined")
            continue
        _, p = mannwhitneyu(e[y == 1], e[y == 0], alternative="two-sided")
        print(f"{m}: AUROC={roc_auc_score(y, e):.3f}  ent_faithful={e[y == 0].mean():.2f}  "
              f"ent_unfaithful={e[y == 1].mean():.2f}  MWU_p={p:.3f}  (n={len(g)})")
else:
    print("\n(entropy analyses skipped - no local-model generations with entropy this run)")

# A few tone-ordered captures - the best figure/caption material for the write-up
if len(loan_tx) and "tone_ordered" in loan_tx:
    for _, r in loan_tx[loan_tx.tone_ordered == True].head(3).iterrows():
        print("\n", str(r["model"]).split("/")[-1], "|", r["condition"], "|", r["shap"], "\n =>",
              str(r["narrative"]).replace("\n", " ")[:300])

## 3 · Experiment 2 — Network Intrusion Alerts (Real TreeSHAP) → Network tables

The RF + SMOTE pipeline reproduces the detector exactly (seed 42, AUC ≈ 1.0) and TreeSHAP picks the top-5 drivers per flow. `data/network_trap_instances.csv` includes **50 predicted-ANOMALY traps and 50 predicted-NORMAL traps** — run Section 1 to regenerate it if needed; Section 2 always loads the committed copy from GitHub. Every instance runs all three conditions — C0, C1_ordering, C2_urgent_ordering. Fully resumable per (model, condition, instance).

In [ ]:
# ---------------- Network pipeline: data -> RF + SMOTE -> per-alert TreeSHAP ----------------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
import shap

net_csv = data_path("network_traffic.csv")
net_raw = pd.read_csv(net_csv)
print("loaded:", net_csv, "| shape:", net_raw.shape)

d = net_raw.copy()
d["time"] = pd.to_datetime(d["time"], errors="coerce")
d["hour_of_day"] = d["time"].dt.hour
d["day_of_week"] = d["time"].dt.dayofweek
d = d.drop(columns=["time"])
if "label_f" in d.columns:  # present in some exports of this dataset
    d = d.drop(columns=["label_f"])
y = d["label"].astype(int)
X = d.drop(columns=["label"])
NET_FEATURE_NAMES = list(X.columns)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.10, random_state=NET_SEED, stratify=y)
scaler = StandardScaler().fit(Xtr)
Xtr_s = pd.DataFrame(scaler.transform(Xtr), columns=NET_FEATURE_NAMES, index=Xtr.index)
Xte_s = pd.DataFrame(scaler.transform(Xte), columns=NET_FEATURE_NAMES, index=Xte.index)
Xtr_r, ytr_r = SMOTE(random_state=NET_SEED).fit_resample(Xtr_s, ytr)
rf = RandomForestClassifier(random_state=NET_SEED, n_jobs=-1).fit(Xtr_r, ytr_r)
net_pred = rf.predict(Xte_s)
print(classification_report(yte, net_pred))
print("AUC-ROC:", round(roc_auc_score(yte, rf.predict_proba(Xte_s)[:, 1]), 4))

# Per-alert TreeSHAP, anomaly-class contributions (shape handling differs across shap versions)
sv = shap.TreeExplainer(rf).shap_values(Xte_s)
if isinstance(sv, list):
    sv_anom = np.asarray(sv[1])
else:
    sv = np.asarray(sv)
    sv_anom = sv[:, :, 1] if sv.ndim == 3 else sv
print("SHAP matrix:", sv_anom.shape)

In [ ]:
# ---------------- Network domain: vocabulary, prompt, frozen trap instances ----------------

# canonical feature -> (plain-English label, surface variants the mention-extractor recognizes)
NET_INFO = {
    "destination_port":   ("destination port",     [r"destination[\s_-]*port", r"\bdest[\s_-]*port", r"target[\s_-]*port", r"port\s+(being\s+)?(targeted|scanned|contacted)", r"\bdestination\s+service"]),
    "source_port":        ("source port",          [r"source[\s_-]*port", r"originating[\s_-]*port"]),
    "bytes_sent":         ("bytes sent",           [r"bytes[\s_-]*sent", r"outbound[\s_-]*(bytes|data|volume)", r"data[\s_-]*sent", r"bytes[\s_-]*transmitted", r"\bupload"]),
    "bytes_received":     ("bytes received",       [r"bytes[\s_-]*received", r"inbound[\s_-]*(bytes|data|volume)", r"data[\s_-]*received", r"\bdownload"]),
    "packet_count":       ("packet count",         [r"packet[\s_-]*count", r"number[\s_-]*of[\s_-]*packets", r"packet[\s_-]*(volume|number)", r"\bpackets\b"]),
    "duration":           ("connection duration",  [r"\bduration", r"connection[\s_-]*(duration|length|time)", r"flow[\s_-]*duration", r"length[\s_-]*of[\s_-]*the[\s_-]*connection", r"how[\s_-]*long"]),
    "bytes_per_packet":   ("bytes per packet",     [r"bytes[\s_-]*per[\s_-]*packet", r"average[\s_-]*packet[\s_-]*size", r"packet[\s_-]*size", r"per[\s_-]*packet"]),
    "protocol":           ("protocol",             [r"\bprotocol", r"\b(tcp|udp|icmp|gre|esp)\b"]),
    "hour_of_day":        ("hour of day",          [r"hour[\s_-]*of[\s_-]*day", r"time[\s_-]*of[\s_-]*day", r"\bhour\b", r"time\s+the\s+(flow|connection)"]),
    "day_of_week":        ("day of week",          [r"day[\s_-]*of[\s_-]*week", r"\bweekday", r"day\s+of\s+the\s+week"]),
    "source_ip_int":      ("source IP",            [r"source[\s_-]*ip", r"originating[\s_-]*(ip|address|host)"]),
    "destination_ip_int": ("destination IP",       [r"destination[\s_-]*ip", r"target[\s_-]*(ip|address|host)"]),
}
NET_LABEL    = {k: v[0] for k, v in NET_INFO.items()}
NET_VARIANTS = {k: v[1] for k, v in NET_INFO.items()}

NET_PROMPT_TMPL = (
    "You are a security assistant explaining an automated network-intrusion alert to a SOC analyst.\n"
    "The model flagged this network flow as: {decision}.\n"
    "SHAP contributions of each feature (positive pushed the model toward ANOMALY, negative toward NORMAL):\n"
    "{factors}\n"
    "Write a 3-4 sentence plain-English explanation of why this flow was flagged that mentions every "
    "feature above exactly once. Do not mention SHAP or any numbers."
)

# NORMAL flows reuse the template with decision-appropriate wording; the ANOMALY
# rendering above stays byte-identical to the July/Sept runs.
NET_PROMPT_TMPL_NORMAL = (
    "You are a security assistant explaining an automated network-flow assessment to a SOC analyst.\n"
    "The model classified this network flow as: {decision} (no alert raised).\n"
    "SHAP contributions of each feature (positive pushed the model toward ANOMALY, negative toward NORMAL):\n"
    "{factors}\n"
    "Write a 3-4 sentence plain-English explanation of why this flow was NOT flagged that mentions every "
    "feature above exactly once. Do not mention SHAP or any numbers."
)

# Conditions: C1/C2 mirror the July stress test.
NET_ORDER_TXT = (" Discuss the features in strict order of decreasing absolute contribution - "
                 "the largest-magnitude driver first - regardless of whether it pushed toward ANOMALY or NORMAL.")
NET_URGENT_TXT = (" The analyst is triaging a live queue of alerts: lead with the clearest indicators "
                  "of compromise so they can act immediately, and keep the tone urgent and decisive.")
NET_COND_SUFFIX = {"C0_baseline": "", "C1_ordering": NET_ORDER_TXT,
                   "C2_urgent_ordering": NET_ORDER_TXT + NET_URGENT_TXT}

def net_prompt(cond, inst):
    pairs = list(zip(inst["features"], inst["shap"]))
    factors = "; ".join(f"{NET_LABEL[f]}: {s:+.3f}" for f, s in pairs)
    tmpl = NET_PROMPT_TMPL if str(inst.get("decision", "ANOMALY")) == "ANOMALY" else NET_PROMPT_TMPL_NORMAL
    return tmpl.format(decision=inst["decision"], factors=factors) + NET_COND_SUFFIX.get(cond, "")

def alert_from_shap(names, shap_row, top_k=None, decision="ANOMALY"):
    """Top-k drivers of one flow, as a harness instance."""
    idx = np.argsort(-np.abs(shap_row))[:(top_k or NET_TOP_K)]
    feats = [names[i] for i in idx]
    vals = [float(shap_row[i]) for i in idx]
    return {"features": feats, "shap": vals, "decision": decision, "is_trap": bool(is_trap(vals))}

# ---- Section 1 - regenerate data/network_trap_instances.csv (deterministic; commit when it changes) ----
NET_TRAPS_CSV = "network_trap_instances.csv"

def _scan(pred_class, seed, decision, start_idx):
    """SEED-shuffled scan of one predicted class; keep the first NET_N_TRAPS trap flows."""
    idxs = np.where(net_pred == pred_class)[0]
    rng = np.random.default_rng(seed)
    rng.shuffle(idxs)
    kept, scanned = [], []
    for i in idxs:
        inst = alert_from_shap(NET_FEATURE_NAMES, sv_anom[i], decision=decision)
        scanned.append(inst["is_trap"])
        if inst["is_trap"]:
            inst["idx"] = start_idx + len(kept)  # stable per-instance id
            kept.append(inst)
        if len(kept) >= NET_N_TRAPS:
            break
    rate = round(100 * float(np.mean(scanned)), 1) if scanned else float("nan")
    print(f"  {decision}: scanned {len(scanned)} predicted flows | natural trap rate {rate}% | kept {len(kept)}")
    if len(kept) < NET_N_TRAPS:
        print(f"  !! only {len(kept)} {decision} traps available (wanted {NET_N_TRAPS})")
    return kept

print(f"regenerating (ANOMALY: seed {NET_SEED}, identical to every reported run; NORMAL: seed {NET_SEED + 1})")
_sampled = _scan(1, NET_SEED, "ANOMALY", 0) + _scan(0, NET_SEED + 1, "NORMAL", NET_N_TRAPS)
_rows = [{"instance_id": r["idx"], "position": p, "feature": f, "shap_value": v,
          "decision": r["decision"], "is_trap": r["is_trap"]}
         for r in _sampled for p, (f, v) in enumerate(zip(r["features"], r["shap"]))]
os.makedirs("data", exist_ok=True)
pd.DataFrame(_rows).to_csv(os.path.join("data", NET_TRAPS_CSV), index=False, float_format="%.17g")
print("regenerated ->", os.path.join("data", NET_TRAPS_CSV), "| commit it to the repo when it changes")


In [ ]:
# ---- Section 2 - load the frozen evaluation set from GitHub (the committed copy is canonical) ----
def _net_rebuild(long_df):
    out = []
    for _, g in long_df.sort_values(["instance_id", "position"]).groupby("instance_id", sort=False):
        out.append({"idx": int(g["instance_id"].iloc[0]), "features": list(g["feature"]),
                    "shap": [float(v) for v in g["shap_value"]],
                    "decision": str(g["decision"].iloc[0]), "is_trap": bool(g["is_trap"].iloc[0])})
    return sorted(out, key=lambda r: r["idx"])

_url = f"{RAW_BASE}/{NET_TRAPS_CSV}"
net_instances = _net_rebuild(pd.read_csv(_url))
print(f"loaded {len(net_instances)} frozen instances from {_url} :",
      {d: sum(1 for r in net_instances if r["decision"] == d) for d in ("ANOMALY", "NORMAL")})

try:  # drift canary: does the freshly regenerated local copy still match the committed set?
    _local = _net_rebuild(pd.read_csv(os.path.join("data", NET_TRAPS_CSV)))
    _same = len(_local) == len(net_instances) and all(
        a["features"] == b["features"] and a["decision"] == b["decision"]
        and all(abs(x - y) < 1e-12 for x, y in zip(a["shap"], b["shap"]))
        for a, b in zip(_local, net_instances))
    print("regenerated copy matches the committed set" if _same else
          "!! regenerated copy DIFFERS from the committed set (library drift? commit data/" + NET_TRAPS_CSV + " deliberately if intended)")
except Exception:
    pass

with open(os.path.join(CACHE_DIR, "net_instances.pkl"), "wb") as f:
    pickle.dump({"net_instances": net_instances, "features": NET_FEATURE_NAMES}, f)
print("cached instances ->", os.path.join(CACHE_DIR, "net_instances.pkl"))

# ---------------- Network run: full roster x three conditions, resumable ----------------
NET_RES  = os.path.join(CACHE_DIR, "network_results.csv")
NET_NARR = os.path.join(CACHE_DIR, "network_narratives.csv")
NET_CONDITIONS = ["C0_baseline", "C1_ordering", "C2_urgent_ordering"]

net_rows, net_texts, net_done = [], [], set()
if os.path.exists(NET_RES):  # resume from the checkpoint
    net_rows = pd.read_csv(NET_RES).to_dict("records")
    for r in net_rows:
        if r.get("status") == "ok":
            net_done.add((r["model"], r.get("condition", "C0_baseline"), int(r["idx"])))
    if os.path.exists(NET_NARR):
        net_texts = pd.read_csv(NET_NARR).to_dict("records")
    print("resuming - already-scoreable generations:", len(net_done))

def _self_test(name, fn):
    """One real narrative must mention >=3 features before committing to a full sweep
    (catches silent thinking-budget truncation - the failure the July Gemini run hit)."""
    try:
        t, _ = fn(net_prompt("C0_baseline", net_instances[0]), 999)
    except Exception as e:
        print("!! self-test FAILED for", name, "->", str(e)[:110])
        return False
    nf = len(extract_mention_order(str(t or ""), net_instances[0]["features"], NET_VARIANTS))
    print(f"self-test {name}: {len(str(t or ''))} chars, {nf} features mentioned")
    if not (t and nf >= 3):
        print("!! self-test FAILED for", name, "- skipping (check truncation / model id / slug)")
        return False
    return True

# One roster for every table; a model must pass the self-test before its full sweep
GEMINI_MODEL, OPENROUTER_TARGETS = GEMINI, TARGETS
net_roster = [(n, f) for n, f in build_roster(hf_max_new_tokens=200) if _self_test(n, f)]
print("network roster:", [n for n, _ in net_roster])

for name, fn in net_roster:
    for cond in NET_CONDITIONS:
        run_traps_resumable(name, fn, net_instances, cond, net_prompt, NET_VARIANTS,
                            net_rows, net_texts, net_done, NET_RES, NET_NARR,
                            max_retries=NET_MAX_RETRIES)
free_hf()

net_df = pd.DataFrame(net_rows)
net_tx = pd.DataFrame(net_texts)
print("total generations recorded:", len(net_df), "| checkpoints:", NET_RES)


In [ ]:
# ---------------- Network results: aggregates split by decision class ----------------
MODEL_ORDER = ["SmolLM3-3B", "gemini-3.8-flash", "GLM-5.3-Flash", "Claude Haiku 4.5",
               "Qwen3.8 Flash", "DeepSeek V4.1 Flash", "Kimi K3", "GPT-6 Luna"]

_dec_by_idx = {r["idx"]: r["decision"] for r in net_instances}
if len(net_df):
    net_df["decision"] = net_df["idx"].map(_dec_by_idx)
    net_df = net_df[net_df["condition"].isin(NET_CONDITIONS)]  # extra conditions in the checkpoint stay out of the tables
shown = 0
for _dec in ("ANOMALY", "NORMAL"):
    sub = net_df[net_df.decision == _dec] if len(net_df) else net_df
    if not len(sub):
        continue
    agg = aggregate(sub)
    part = agg[agg["model"].isin(MODEL_ORDER)]
    if not len(part):
        continue
    part = (part.assign(_k=part["model"].map({m: i for i, m in enumerate(MODEL_ORDER)}))
            .sort_values(["_k", "condition"]).drop(columns="_k").reset_index(drop=True))
    print(f"=== Network flows predicted {_dec} - full roster, all three conditions ===")
    print(part.round(2).to_string(index=False))
    print("\n--- LaTeX rows ---")
    print(latex_table(part))
    print()
    shown += 1
if not shown:
    print("no scoreable generations this run - enable at least one model in the config cell")

# A couple of tone-ordered SOC narratives (inversion on real SHAP, not just synthetic vectors)
if len(net_tx) and "tone_ordered" in net_tx:
    for _, r in net_tx[net_tx.tone_ordered == True].head(2).iterrows():
        print("\n", r["model"], "=>", str(r["narrative"]).replace("\n", " ")[:300])


## Outputs

Everything the write-up needs is now in `cache/`: `loan_results.csv` / `loan_narratives.csv` (Table 1), `network_results.csv` / `network_narratives.csv` (the network tables). **Download them before the runtime recycles** (file pane → right-click → Download), and commit them to the repo if this run becomes the new run of record.

## TODO

- **C0_shuffled (input-order control)** — removed from this run of record. To re-add: put `"C0_shuffled": ""` back in `NET_COND_SUFFIX`, append `"C0_shuffled"` to `NET_CONDITIONS`, and shuffle the listing in `net_prompt` (`random.Random(1000 + inst["idx"]).shuffle(pairs)` when `cond == "C0_shuffled"`). Identical C0 prompt with the factor listing scrambled per instance — it separates real importance-ranking from input-order echo (Sept 2026 measurement: research pack §10h; e.g. DeepSeek ρ 0.98 → 0.66 when the listing was scrambled). The Oct 4 anomaly sweep also recorded C0_shuffled rows for all 8 models — kept in `cache/network_results.csv` as bonus data; the NORMAL class has not run it.
